# Timeseries Anomaly Explanation with LLMs

## Overview

This notebook demonstrates a pipeline for **identifying, contextualizing, and explaining anomalies** in timeseries indicator data using Large Language Models (LLMs). The approach is **generalized** to work with any timeseries dataset (World Development Indicators, Corporate Scorecard, or custom indicators)—not just a single application.

### Purpose

Automatically generate verifiable explanations for significant data deviations, classifying them as:

- **External drivers** — macroeconomic events, conflicts, policy reforms, disasters
- **Data errors** — placeholders, ingestion issues, computation artifacts
- **Measurement system updates** — rebasing, census revisions, classification changes
- **Insufficient data** — when no verifiable cause can be identified

### Learning Objectives

1. The **canonical data interface** for timeseries anomaly inputs
2. How to **adapt legacy formats** (e.g., Scorecard wide format) to the canonical schema
3. The **context extraction** logic for building LLM prompts
4. How **structured prompts** and JSON schema constrain LLM outputs
5. How to **parse and analyze** batch LLM outputs

### Prerequisites

- Python 3.11+ with `ai4data[anomaly]` and `ai4data[metadata]` installed
- API keys for OpenAI and/or Gemini (for LLM inference)

## Pipeline Overview

```mermaid
flowchart LR
    subgraph input [Input]
        A[Wide CSV]
        B[Anomaly Scores CSV]
    end
    subgraph adapt [Transform]
        C[Adapter]
    end
    subgraph process [Pipeline]
        D[Canonical Format]
        E[Anomaly Ranking]
        F[Context Extraction]
        G[LLM Prompts]
        H[Batch API]
        I[Parsed Output]
    end
    A --> C
    B --> C
    C --> D
    D --> E
    E --> F
    F --> G
    G --> H
    H --> I
```

## Step 1: Understand the Data Interface

The pipeline consumes data in a **canonical long-format** schema. This format is designed to be:

- **Indicator-agnostic** — works for WDI, Scorecard, or custom indicators
- **Geography-agnostic** — countries, regions, or any spatial unit
- **Explicit about anomaly metadata** — pre-computed scores and imputation flags

### Canonical Column Semantics

| Column | Type | Description |
|--------|------|-------------|
| `indicator_id` | str | Unique indicator code (e.g., `WB_CSC_SI_POV_UMIC`) |
| `indicator_name` | str | Human-readable indicator label |
| `geography_id` | str | Geography code (e.g., `ISL` for Iceland) |
| `geography_name` | str | Human-readable geography label |
| `period` | int | Time period (year for annual data) |
| `value` | float | Indicator value (NaN allowed) |
| `is_imputed` | bool | Whether the value was imputed |
| `anomaly_score` | float | Pre-computed anomaly magnitude (e.g., \|z-score\|) |
| `outlier_count` | int | Number of detectors that flagged this point |

## Step 2: Load and Adapt Input Data

If your data is in a legacy format (e.g., Scorecard wide format with separate anomaly scores), use the **adapter** to convert to canonical format.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from jinja2 import Template

from ai4data.anomaly_detection import (
    ScorecardWideAdapter,
    extract_anomaly_contexts,
    parse_batch_output,
)
from ai4data.anomaly_detection.llm_client import build_payload, ENDPOINT_URLS
from ai4data.anomaly_detection.prompts import (
    SYSTEM_PROMPT,
    USER_PROMPT_TEMPLATE,
    get_anomaly_response_format,
)

In [ ]:
# Configuration: adjust paths to your data
# For Scorecard: wide-format CSV + anomaly scores CSV
DATA_DIR = Path(os.environ.get("ANOMALY_DATA_DIR", "."))
WIDE_PATH = DATA_DIR / "WB_CSC_WIDEF.csv"
ANOMALY_PATH = DATA_DIR / "CSC_TOP_ANOMALIES_2026-02-06.CSV"

# If files exist, load and adapt
if WIDE_PATH.exists() and ANOMALY_PATH.exists():
    adapter = ScorecardWideAdapter()
    canonical_df = adapter.load(WIDE_PATH, ANOMALY_PATH)
    print(f"Loaded {len(canonical_df)} rows")
    display(canonical_df.head())
else:
    print("Data files not found. Create sample data or set ANOMALY_DATA_DIR.")
    canonical_df = pd.DataFrame()

## Step 3: Anomaly Identification

We filter for series with multiple anomaly detectors in agreement (`outlier_count >= 3`) and rank by combined `anomaly_score`. This produces a shortlist of (indicator, geography) pairs to explain.

In [ ]:
shortlist = pd.DataFrame()
if len(canonical_df) > 0:
    # Filter: at least 3 detectors agree, exclude imputed
    filtered = canonical_df[
        (canonical_df["outlier_count"] >= 3) & (~canonical_df["is_imputed"])
    ]
    # Rank by sum of anomaly_score per (indicator, geography)
    ranked = (
        filtered.groupby(["indicator_id", "geography_id"])["anomaly_score"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    shortlist = ranked.head(10_000)  # Cap for batch processing
    print(f"Shortlist: {len(shortlist)} series")
    display(shortlist.head())

## Step 4: Context Generation

For each (indicator, geography) in the shortlist, we extract a **time-windowed context** around the anomaly years. Overlapping windows are merged into contiguous ranges so the LLM sees a single coherent snippet per anomaly cluster.

In [ ]:
if len(canonical_df) > 0 and len(shortlist) > 0:
    # Build name maps for context
    geo_map = canonical_df.set_index("geography_id")["geography_name"].to_dict()
    ind_map = canonical_df.set_index("indicator_id")["indicator_name"].to_dict()

    # Index by (indicator_id, geography_id) for fast lookup
    source_df = canonical_df.set_index(["indicator_id", "geography_id"]).sort_index()

    # Extract one example context
    row = shortlist.iloc[0]
    ind_id, geo_id = row["indicator_id"], row["geography_id"]
    series_df = source_df.loc[(ind_id, geo_id)].reset_index()
    contexts = extract_anomaly_contexts(
        series_df,
        geography_name_map=geo_map,
        indicator_name_map=ind_map,
        period_window=3,
        min_outlier_count=3,
    )
    print("Example context (first series):")
    print(json.dumps(contexts[0] if contexts else {}, indent=2))

## Step 5: LLM Prompting

The prompt design instructs the LLM to:
1. Treat anomalies as **windows** (start, end), not single points
2. Classify into one of five categories
3. Provide **evidence strength** and optional **evidence sources**
4. Output strictly valid JSON matching the schema

In [ ]:
user_template = Template(USER_PROMPT_TEMPLATE)
response_format = get_anomaly_response_format()

# Example: render prompt with a context
if contexts:
    sample_context = json.dumps(contexts[0], indent=2)
    sample_prompt = user_template.render(INPUT_SERIES_INFO=sample_context)
    print("Sample user prompt (first 1500 chars):")
    print(sample_prompt[:1500] + "...")

## Step 6: LLM Inference

To run batch inference:
1. Build a JSONL file of requests (one per context)
2. Upload to OpenAI Batch API or Gemini File API
3. Poll for completion and download results

For this notebook, we show how to **build the batch file**. Running the actual batch requires an API key and is typically done separately.

In [ ]:
from hashlib import md5
from datetime import datetime, timezone

# Build batch JSONL (example: first 5 series only for demo)
OUTPUT_DIR = Path(os.environ.get("ANOMALY_OUTPUT_DIR", "./llm-output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_id = "gpt-4.1-mini"
endpoint = "responses"
prompt_hash = md5((SYSTEM_PROMPT + "\n" + USER_PROMPT_TEMPLATE).encode()).hexdigest()[:8]
batch_fname = OUTPUT_DIR / f"openai-anomaly-{int(datetime.now(tz=timezone.utc).timestamp())}.jsonl"

if len(shortlist) > 0:
    count = 0
    with open(batch_fname, "w") as f:
        for _, row in shortlist.head(5).iterrows():  # Limit to 5 for demo
            ind_id, geo_id = row["indicator_id"], row["geography_id"]
            series_df = source_df.loc[(ind_id, geo_id)].reset_index()
            ctxs = extract_anomaly_contexts(series_df, geo_map, ind_map, period_window=3, min_outlier_count=3)
            for ctx in ctxs:
                ctx_str = json.dumps(ctx, indent=2)
                user_prompt = user_template.render(INPUT_SERIES_INFO=ctx_str)
                payload = build_payload(
                    endpoint=endpoint,
                    model_id=model_id,
                    system_prompt=SYSTEM_PROMPT,
                    user_prompt=user_prompt,
                    response_format=response_format,
                    with_search=False,
                )
                custom_id = f"nosearch-{prompt_hash}-{ind_id}-{geo_id}-{md5(ctx_str.encode()).hexdigest()}"
                batch_row = {
                    "custom_id": custom_id,
                    "method": "POST",
                    "url": ENDPOINT_URLS[endpoint],
                    "body": payload,
                }
                f.write(json.dumps(batch_row) + "\n")
                count += 1
    print(f"Wrote {count} requests to {batch_fname}")

## Step 7: Output Analysis

Parse the batch output JSONL into a DataFrame for analysis and export.

In [ ]:
# Parse batch output (replace with your actual output path)
ind_map = globals().get("ind_map", {})
geo_map = globals().get("geo_map", {})
output_path = OUTPUT_DIR / "openai-anomaly-output.jsonl"
if output_path.exists():
    anomalies_df = parse_batch_output(
        output_path,
        provider="openai",
        indicator_name_map=ind_map,
        geography_name_map=geo_map,
        custom_id_parts=(0, 2, 3),  # prefix, indicator_idx, geography_idx in custom_id
    )
    display(anomalies_df.head())
    anomalies_df.groupby("classification").agg(
        count=("confidence", "count"),
        mean_conf=("confidence", "mean"),
    )
else:
    print("Batch output not found. Run inference and save to", output_path)

## Appendix: Using Your Own Data

For custom data with different column names, use `adapter_from_config`:

```python
from ai4data.anomaly_detection import adapter_from_config

mapping = {
    "indicator_id": "YOUR_INDICATOR_COL",
    "indicator_name": "YOUR_INDICATOR_LABEL_COL",
    "geography_id": "YOUR_GEO_COL",
    "geography_name": "YOUR_GEO_LABEL_COL",
    "period": "YEAR",
    "value": "VALUE",
    "is_imputed": "Imputed",
    "anomaly_score": "absZscore",  # or create from Zscore
    "outlier_count": "outlier_indicator_total",
}
adapt = adapter_from_config(mapping)
canonical_df = adapt("wide.csv", "anomalies.csv")
```